In [ ]:
import os
import math
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

# from  import Path
from torch.utils.data import Dataset, DataLoader
from tokenizers import ByteLevelBPETokenizer

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Device
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Shared configuration
BLOCK_SIZE = 64
BATCH_SIZE = 32
LEARNING_RATE = 0.001
EPOCHS = 10

print("PyTorch:", torch.__version__)
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.14.1+cpu
Device: cpu


Loading Tokenizer

In [28]:
PROJECT_DIR = Path.cwd()

TOKENIZER_DIR = PROJECT_DIR / "models" / "krish_bpe"

VOCAB_PATH = TOKENIZER_DIR / "vocab.json"
MERGES_PATH = TOKENIZER_DIR / "merges.txt"

assert VOCAB_PATH.exists(), f"Missing: {VOCAB_PATH}"
assert MERGES_PATH.exists(), f"Missing: {MERGES_PATH}"

tokenizer = ByteLevelBPETokenizer(
    str(VOCAB_PATH),
    str(MERGES_PATH)
)

VOCAB_SIZE = tokenizer.get_vocab_size()
EOS_ID = tokenizer.token_to_id("<|eos|>")

assert EOS_ID is not None, "EOS token not found!"

print("Tokenizer loaded successfully!")
print("Vocabulary size:", VOCAB_SIZE)
print("EOS ID:", EOS_ID)

Tokenizer loaded successfully!
Vocabulary size: 2000
EOS ID: 3


In [30]:
# Use the actual column names from your CSV

TEXT_COLUMN = "text"

messages = (
    girl_df[TEXT_COLUMN]
    .dropna()
    .astype(str)
)

messages = messages[
    messages.str.strip() != ""
].tolist()

print("Total text messages:", len(messages))

print("\nFirst 5 messages:")
for message in messages[:5]:
    print(message)

Total text messages: 55623

First 5 messages:
Koi baat nhi
Tumne msg kya kiya tha?
Happy Republic Day 😊
Btw
Same to u


Dividing the Messages into Train , validation and test split 

In [31]:
total_messages = len(messages)

train_end = int(total_messages * 0.90)
val_end = int(total_messages * 0.95)

train_messages = messages[:train_end]
val_messages = messages[train_end:val_end]
test_messages = messages[val_end:]

print("Training messages:", len(train_messages))
print("Validation messages:", len(val_messages))
print("Testing messages:", len(test_messages))
print("Total:", len(train_messages) + len(val_messages) + len(test_messages))

Training messages: 50060
Validation messages: 2781
Testing messages: 2782
Total: 55623


In [32]:
def encode_messages(messages, tokenizer):

    all_ids = []

    eos_id = tokenizer.token_to_id("<|eos|>")

    for message in messages:

        token_ids = tokenizer.encode(message).ids

        all_ids.extend(token_ids)
        all_ids.append(eos_id)

    return torch.tensor(all_ids, dtype=torch.long)


train_ids = encode_messages(train_messages, tokenizer)
val_ids = encode_messages(val_messages, tokenizer)
test_ids = encode_messages(test_messages, tokenizer)

print("Training token IDs:", len(train_ids))
print("Validation token IDs:", len(val_ids))
print("Testing token IDs:", len(test_ids))

print("\nFirst 20 training IDs:")
print(train_ids[:20].tolist())

Training token IDs: 391531
Validation token IDs: 19513
Testing token IDs: 19832

First 20 training IDs:
[760, 346, 286, 3, 1598, 289, 617, 347, 503, 343, 34, 3, 446, 862, 92, 224, 1909, 83, 1363, 391]


In [33]:
BLOCK_SIZE = 64

class NextTokenDataset(Dataset):

    def __init__(self, token_ids, block_size):
        self.token_ids = token_ids
        self.block_size = block_size

    def __len__(self):
        return len(self.token_ids) - self.block_size

    def __getitem__(self, idx):

        x = self.token_ids[idx : idx + self.block_size]

        y = self.token_ids[idx + 1 : idx + self.block_size + 1]

        return x, y


train_dataset = NextTokenDataset(train_ids, BLOCK_SIZE)
val_dataset = NextTokenDataset(val_ids, BLOCK_SIZE)
test_dataset = NextTokenDataset(test_ids, BLOCK_SIZE)

print("Training sequences:", len(train_dataset))
print("Validation sequences:", len(val_dataset))
print("Testing sequences:", len(test_dataset))

Training sequences: 391467
Validation sequences: 19449
Testing sequences: 19768


DataLoaders

In [34]:
BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Testing batches:", len(test_loader))

Training batches: 12234
Validation batches: 608
Testing batches: 618


In [35]:
x, y = next(iter(train_loader))

print("Input shape:", x.shape)
print("Target shape:", y.shape)

print("\nFirst input sequence:")
print(x[0].tolist())

print("\nFirst target sequence:")
print(y[0].tolist())

print("\nDecoded input:")
print(tokenizer.decode(x[0].tolist()))

print("\nDecoded target:")
print(tokenizer.decode(y[0].tolist()))

Input shape: torch.Size([32, 64])
Target shape: torch.Size([32, 64])

First input sequence:
[298, 583, 3, 664, 710, 284, 294, 984, 422, 79, 913, 492, 1228, 1871, 3, 1882, 1111, 1087, 379, 314, 1337, 322, 290, 80, 3, 429, 287, 1201, 383, 78, 283, 303, 1834, 268, 3, 648, 268, 3, 999, 523, 475, 1991, 288, 377, 287, 401, 1192, 3, 382, 554, 3, 55, 68, 372, 326, 492, 525, 917, 1055, 288, 434, 88, 1235, 3]

First target sequence:
[583, 3, 664, 710, 284, 294, 984, 422, 79, 913, 492, 1228, 1871, 3, 1882, 1111, 1087, 379, 314, 1337, 322, 290, 80, 3, 429, 287, 1201, 383, 78, 283, 303, 1834, 268, 3, 648, 268, 3, 999, 523, 475, 1991, 288, 377, 287, 401, 1192, 3, 382, 554, 3, 55, 68, 372, 326, 492, 525, 917, 1055, 288, 434, 88, 1235, 3, 278]

Decoded input:
 ho jaye<|eos|>Abhi itna dur travel karega tujhe jarurat padegi<|eos|>kabhi alwida na kehna sanam<|eos|>abhi to mulakat bhi baaki hai<|eos|>Thik hai<|eos|>Jab recharge kar le to bata diyo<|eos|>Achha sun<|eos|>Taki mai tujhe call karke pareshan k

Bigram model 

In [37]:

# ============================================================
# 1. BUILD BIGRAM TRANSITION COUNTS
# ============================================================

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import math
from pathlib import Path

# Add smoothing to handle unseen token transitions
ALPHA = 0.1

# Each cell [i, j] stores how often token j follows token i
transition_counts = torch.zeros(
    (VOCAB_SIZE, VOCAB_SIZE),
    dtype=torch.float64
)

# Count adjacent token pairs from training data
current_tokens = train_ids[:-1]
next_tokens = train_ids[1:]

pair_indices = (
    current_tokens * VOCAB_SIZE + next_tokens
)

counts = torch.bincount(
    pair_indices,
    minlength=VOCAB_SIZE * VOCAB_SIZE
)

transition_counts = counts.reshape(
    VOCAB_SIZE, VOCAB_SIZE
).double()

print("Transition counts created!")
print("Matrix shape:", transition_counts.shape)
print("Total transitions:", int(transition_counts.sum()))


# ============================================================
# 2. CALCULATE NEXT-TOKEN PROBABILITIES
# ============================================================

smoothed_counts = transition_counts + ALPHA

transition_probs = (
    smoothed_counts /
    smoothed_counts.sum(dim=1, keepdim=True)
)

# Store log probabilities for efficient evaluation
log_probs = torch.log(transition_probs).to(device)

print("Probability matrix shape:", transition_probs.shape)
print("Probability matrix ready!")


# ============================================================
# 3. EVALUATION FUNCTION
# ============================================================

def evaluate_bigram(token_ids):

    token_ids = token_ids.to(device)

    x = token_ids[:-1]
    y = token_ids[1:]

    # Select probability of the actual next token
    selected_log_probs = log_probs[x, y]

    loss = -selected_log_probs.mean()

    # Most likely next-token prediction
    predictions = log_probs[x].argmax(dim=1)

    accuracy = (predictions == y).float().mean()

    perplexity = torch.exp(loss)

    return {
        "loss": loss.item(),
        "accuracy": accuracy.item(),
        "perplexity": perplexity.item()
    }


# ============================================================
# 4. VALIDATION
# ============================================================

val_results = evaluate_bigram(val_ids)

print("\nVALIDATION RESULTS")
print("----------------------")
print("Loss:", round(val_results["loss"], 4))
print("Accuracy:", round(val_results["accuracy"], 4))
print("Perplexity:", round(val_results["perplexity"], 4))


# ============================================================
# 5. FINAL TEST EVALUATION
# ============================================================

test_results = evaluate_bigram(test_ids)

print("\nTEST RESULTS")
print("----------------------")
print("Loss:", round(test_results["loss"], 4))
print("Accuracy:", round(test_results["accuracy"], 4))
print("Perplexity:", round(test_results["perplexity"], 4))


# ============================================================
# 6. GENERATE TEXT
# ============================================================

def generate_bigram(
    prompt,
    max_new_tokens=50,
    temperature=1.0
):

    if temperature <= 0:
        raise ValueError("Temperature must be positive.")

    model_probs = transition_probs

    encoded = tokenizer.encode(prompt).ids

    if not encoded:
        raise ValueError("Prompt cannot be empty.")

    generated = encoded.copy()

    for _ in range(max_new_tokens):

        current_token = generated[-1]

        probabilities = model_probs[current_token]

        # Temperature sampling
        probabilities = torch.softmax(
            torch.log(probabilities) / temperature,
            dim=0
        )

        next_token = torch.multinomial(
            probabilities.float(),
            num_samples=1
        ).item()

        if next_token == EOS_ID:
            break

        generated.append(next_token)

    return tokenizer.decode(generated)


# ============================================================
# 7. GENERATE SAMPLE TEXT
# ============================================================

sample = generate_bigram(
    prompt="I",
    max_new_tokens=50,
    temperature=0.8
)

print("\nGENERATED TEXT")
print("----------------------")
print(sample)


# ============================================================
# 8. SAVE BIGRAM MODEL
# ============================================================

SAVE_DIR = Path("models/trained")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

torch.save(
    {
        "transition_counts": transition_counts,
        "transition_probs": transition_probs,
        "vocab_size": VOCAB_SIZE,
        "alpha": ALPHA,
        "test_results": test_results
    },
    SAVE_DIR / "bigram_model.pth"
)

print("\nBigram model saved successfully!")


# ============================================================
# 9. VISUALIZE MOST LIKELY NEXT TOKENS
# ============================================================

def show_next_token_predictions(token, top_k=10):

    token_ids = tokenizer.encode(token).ids

    if not token_ids:
        print("Token not recognized.")
        return

    current_id = token_ids[-1]

    probabilities = transition_probs[current_id]

    top_probs, top_ids = torch.topk(
        probabilities,
        k=top_k
    )

    print(f"\nMost likely tokens after: {token}")

    for token_id, probability in zip(
        top_ids.tolist(),
        top_probs.tolist()
    ):

        decoded = tokenizer.decode([token_id])

        print(
            repr(decoded),
            "Probability:",
            round(probability, 4)
        )


show_next_token_predictions("I", top_k=10)


print("\nBIGRAM EXPERIMENT COMPLETED!")


Transition counts created!
Matrix shape: torch.Size([2000, 2000])
Total transitions: 391530
Probability matrix shape: torch.Size([2000, 2000])
Probability matrix ready!

VALIDATION RESULTS
----------------------
Loss: 4.9132
Accuracy: 0.1932
Perplexity: 136.0762

TEST RESULTS
----------------------
Loss: 4.8734
Accuracy: 0.1955
Perplexity: 130.7698

GENERATED TEXT
----------------------
I miss rahungitab🥺🥺Nikal bf is no�

Bigram model saved successfully!

Most likely tokens after: I
' love' Probability: 0.2716
"'m" Probability: 0.0567
' miss' Probability: 0.0326
' mean' Probability: 0.0316
'n' Probability: 0.0253
' hope' Probability: 0.0232
' ' Probability: 0.019
'ss' Probability: 0.0179
'dhar' Probability: 0.0169
' w' Probability: 0.0169

BIGRAM EXPERIMENT COMPLETED!


## LSTM

In [38]:
class LSTMLanguageModel(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim=128,
        hidden_dim=128,
        num_layers=1
    ):
        super().__init__()

        # Token IDs -> embedding vectors
        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim
        )

        # Recurrent LSTM
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True
        )

        # LSTM output -> vocabulary scores
        self.fc = nn.Linear(
            hidden_dim,
            vocab_size
        )

    def forward(self, x):

        x = self.embedding(x)

        output, (hidden_state, cell_state) = self.lstm(x)

        logits = self.fc(output)

        return logits